# M28 lab solution: Grader loop

**Brief.** Add a relevance grader and a two-retry rewrite loop to your Part V system and log how often each path fires on the golden set.

The graph is the one from lesson 28.1 with the retry cap of lesson 28.3: retrieve, grade, and either answer or rewrite the query and retrieve again, at most twice. When the second rewrite still fails the grade, the graph refuses instead of answering from passages it judged irrelevant.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## The Part V system, rebuilt inline

This lab does not depend on another lab's output, so the cell below rebuilds the minimal Part V system: the handbook loaded with `lab_kit`, the baseline chunker (500 characters, 50 overlap), an in-memory vector store over `get_embeddings()`, and an answer prompt that cites passages by number. Later cells reuse `retriever`, `format_docs` and `rag_answer`.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

handbook = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"]}) for p in handbook for c in splitter.split_text(p["text"])]
embeddings = get_embeddings()
store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the Larkspur Cloud handbook. Use only the numbered passages. "
               "Cite each claim with its passage number in square brackets, like [2]. "
               "If the passages do not contain the answer, reply: The handbook does not say."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata['source']}) {d.page_content}" for i, d in enumerate(docs, 1))


def rag_answer(question, docs=None):
    """Retrieve (unless docs are given) and answer with citations; returns (answer, docs)."""
    docs = retriever.invoke(question) if docs is None else docs
    return answer_chain.invoke({"context": format_docs(docs), "question": question}), docs


print(len(chunks), "chunks;", len(golden), "golden questions")

## 1. The grader and the rewriter

In [ ]:
from pydantic import BaseModel, Field


class RelevanceGrade(BaseModel):
    relevant: bool = Field(description="True if the passages contain the information needed to answer the question")


GRADE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You grade retrieval. Say whether the numbered passages contain the information needed to answer the question. "
               "Related passages that do not answer it are not relevant."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
grader = GRADE_PROMPT | llm.with_structured_output(RelevanceGrade)


REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the search query so that it is more likely to find the answer in a cloud provider's customer handbook. "
               "Use the words a handbook would use (billing, refund, backup, region, plan). Return only the new query."),
    ("human", "Original question: {question}\nQuery that found nothing useful: {query}"),
])
rewriter = REWRITE_PROMPT | llm | StrOutputParser()

## 2. The graph

The retry count lives in the state and the routing function reads it, so the cap holds in code whatever the model does.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

MAX_RETRIES = 2
REFUSAL = "The handbook does not say."


class GraderState(TypedDict):
    question: str
    query: str
    docs: list
    relevant: bool
    retries: int
    path: list[str]
    answer: str


def retrieve(state):
    return {"docs": retriever.invoke(state["query"]), "path": state["path"] + ["retrieve"]}


def grade(state):
    g = grader.invoke({"context": format_docs(state["docs"]), "question": state["question"]})
    return {"relevant": g.relevant, "path": state["path"] + ["relevant" if g.relevant else "not relevant"]}


def rewrite(state):
    query = rewriter.invoke({"question": state["question"], "query": state["query"]}).strip()
    return {"query": query, "retries": state["retries"] + 1, "path": state["path"] + [f"rewrite: {query}"]}


def generate(state):
    text, _ = rag_answer(state["question"], state["docs"])
    return {"answer": text, "path": state["path"] + ["answer"]}


def refuse(state):
    return {"answer": REFUSAL, "path": state["path"] + ["refuse"]}


def after_grade(state):
    if state["relevant"]:
        return "generate"
    return "rewrite" if state["retries"] < MAX_RETRIES else "refuse"


builder = StateGraph(GraderState)
for name, fn in [("retrieve", retrieve), ("grade", grade), ("rewrite", rewrite), ("generate", generate), ("refuse", refuse)]:
    builder.add_node(name, fn)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", after_grade, ["generate", "rewrite", "refuse"])
builder.add_edge("rewrite", "retrieve")
builder.add_edge("generate", END)
builder.add_edge("refuse", END)
graph = builder.compile()
print(graph.get_graph().draw_mermaid())

## 3. The 30 golden questions

In [ ]:
PATHS = {0: "answered, first retrieval", 1: "answered after 1 rewrite", 2: "answered after 2 rewrites"}
rows = []
for g in golden:
    out = graph.invoke({"question": g["question"], "query": g["question"], "docs": [], "relevant": False,
                        "retries": 0, "path": [], "answer": ""})
    refused = out["path"][-1] == "refuse"
    hit = any(is_hit(d.page_content, g["evidence_spans"]) for d in out["docs"]) if g["evidence_spans"] else None
    rows.append({"id": g["id"], "type": g["type"], "rewrites": out["retries"],
                 "path": "refused after 2 rewrites" if refused else PATHS[out["retries"]],
                 "final docs hit": hit, "answer": out["answer"], "trace": " > ".join(out["path"])})
log = pd.DataFrame(rows)
print(log[["id", "type", "path", "final docs hit"]].to_string(index=False))

## 4. Results

In [ ]:
counts = pd.crosstab(log.path, log.type, margins=True, margins_name="all")
print(counts.to_string())
print("\nRewritten queries:")
for _, r in log[log.rewrites > 0].iterrows():
    print(f"  {r.id} ({r.type}): {r.trace}")

## What this shows

- The crosstab is the brief's log: how often each path fired, split by question type. Out-of-corpus questions should end in a refusal after two rewrites; an answerable question that does the same is a retrieval miss the rewrites could not fix.
- Every rewrite costs one rewrite call, one retrieval and one more grade. The trace column shows what each rewrite changed, which is where to look before raising the cap.
- A question answered on the first retrieval whose final docs have no evidence span (`final docs hit` False) is a grader error in the other direction: it passed passages that do not answer.